# BRAND Study: Metilene Galaxy Preparation

**Author:** Faith Ogundimu  
**Date:** February 2026  

---

## Purpose

Prepare Bismark bedGraph files for metilene DMR analysis on Galaxy.  
Following the QUB TWIST pipeline document.

**Steps:**
1. Load metadata → classify patients as responder (PFS ≥ 6 months) or non-responder
2. Unzip bedGraph files
3. Clean: remove headers and non-canonical chromosomes
4. Save as `.txt` text files (Galaxy upload format)
5. Organise into group folders for the 5 metilene runs

**Metilene Runs:**

| Run | Group 1 (test) | Group 2 (control) | Question |
|-----|----------------|-------------------|----------|
| 1 | Responder follow-up | Responder baseline | Change in responders |
| 2 | Non-responder follow-up | Non-responder baseline | Change in non-responders |
| 3 | Responder baseline | Non-responder baseline | Groups different at start? |
| 4 | Responder follow-up | Non-responder follow-up | Groups different after treatment? |
| 5 | Sensitivity: Runs 1 & 2 with 6-week follow-up only | - | Stability check |

---

## Section 1: Setup

In [ ]:
import os
import gzip
import shutil
import re
import pandas as pd
from glob import glob
from pathlib import Path

# =============================================================================
# CONFIGURATION - UPDATE THESE PATHS
# =============================================================================

# Root project directory
PROJECT_DIR = Path('../../')  # adjust if notebook is in notebooks/Metilene/

# Input: where your .bedGraph.gz files live
BEDGRAPH_DIR = PROJECT_DIR / 'database_files' / 'bedgraph_files'

# Input: metadata files
CLINICAL_CSV = PROJECT_DIR / 'database_files' / 'metadata' / 'BRAND_merged_analysis_dataset.csv'
SAMPLE_MAP_CSV = PROJECT_DIR / 'database_files' / 'metadata' / 'sample_mapping.csv'

# Output: where cleaned files and group folders go
OUTPUT_DIR = PROJECT_DIR / 'results' / 'metilene'
CLEANED_DIR = OUTPUT_DIR / 'cleaned_bedgraphs'
GROUPS_DIR = OUTPUT_DIR / 'galaxy_groups'

# Responder threshold
PFS_THRESHOLD = 6  # months

# Canonical chromosomes only
CANONICAL_CHROMS = {f'chr{i}' for i in range(1, 23)}
# This is the set of canonical chromosomes (chr1–chr22) that we want to keep for analysis. 
# It excludes sex chromosomes and any non-standard contigs, ensuring a cleaner dataset focused on the main genomic regions.

print(f'Bedgraph dir:  {BEDGRAPH_DIR}')
print(f'Clinical CSV:  {CLINICAL_CSV}')
print(f'Sample map:    {SAMPLE_MAP_CSV}')
print(f'Output dir:    {OUTPUT_DIR}')

## Section 2: Load Metadata and Classify Response Groups

In [ ]:
# =============================================================================
# LOAD CLINICAL DATA
# =============================================================================

clinical = pd.read_csv(CLINICAL_CSV)
sample_map = pd.read_csv(SAMPLE_MAP_CSV)

print(f'Clinical data: {len(clinical)} patients')
print(f'Sample mapping: {len(sample_map)} samples')

# Classify responders
clinical['response'] = clinical['PFS Months'].apply(
    lambda x: 'responder' if x >= PFS_THRESHOLD else 'nonresponder'
)

n_resp = (clinical['response'] == 'responder').sum()
n_nonresp = (clinical['response'] == 'nonresponder').sum()
print(f'\nResponders (PFS >= {PFS_THRESHOLD}mo): {n_resp}')
print(f'Non-responders (PFS < {PFS_THRESHOLD}mo): {n_nonresp}')

In [ ]:
# =============================================================================
# BUILD SAMPLE-LEVEL METADATA
# =============================================================================
# Each patient has 2 samples (baseline + follow-up)
# We need: sample_id -> patient_id -> response group + timepoint

rows = []
for _, pt in clinical.iterrows():
    pid = pt['Patient_ID']
    response = pt['response']
    pfs = pt['PFS Months']
    
    # Baseline sample
    b_id = pt['Baseline_Sample_ID']  # e.g., PN0634_087
    b_num = b_id.replace('PN0634_', '')  # e.g., 087
    rows.append({
        'patient_id': pid,
        'sample_id': b_id,
        'sample_num': int(b_num),
        'timepoint': 'baseline',
        'response': response,
        'pfs_months': pfs
    })
    
    # Follow-up sample
    f_id = pt['Followup_Sample_ID']  # e.g., PN0634_088
    f_num = f_id.replace('PN0634_', '')  # e.g., 088
    rows.append({
        'patient_id': pid,
        'sample_id': f_id,
        'sample_num': int(f_num),
        'timepoint': 'followup',
        'response': response,
        'pfs_months': pfs
    })

metadata = pd.DataFrame(rows)
metadata['group'] = metadata['response'] + '_' + metadata['timepoint']

# Remove the patient whose sequencing failed
EXCLUDED_PATIENTS = ['SJH15']
metadata = metadata[~metadata['patient_id'].isin(EXCLUDED_PATIENTS)]
print(f'Excluded {len(EXCLUDED_PATIENTS)} patient(s) with failed sequencing: {EXCLUDED_PATIENTS}')

print(f'Total samples: {len(metadata)}')
print(f'\nGroup counts:')
print(metadata['group'].value_counts().to_string())

## Section 3: Find and Match BedGraph Files

In [ ]:
# =============================================================================
# FIND BEDGRAPH FILES
# =============================================================================
# Bismark bedGraph files are typically named:
#   PN0634_XXX_SXXX_L001_R1_001_bismark_bt2_pe.deduplicated.bedGraph.gz
# We extract the sample number (XXX) to match to metadata.

bedgraph_files = sorted(glob(str(BEDGRAPH_DIR / '*.bedGraph.gz')))

# If none found, try alternative extensions
if not bedgraph_files:
    bedgraph_files = sorted(glob(str(BEDGRAPH_DIR / '*.bedgraph.gz')))
if not bedgraph_files:
    bedgraph_files = sorted(glob(str(BEDGRAPH_DIR / '*.bedGraph')))
if not bedgraph_files:
    # Maybe they're already unzipped .txt files
    bedgraph_files = sorted(glob(str(BEDGRAPH_DIR / '*.txt')))

print(f'Found {len(bedgraph_files)} bedGraph files')
if bedgraph_files:
    print(f'Example: {os.path.basename(bedgraph_files[0])}')
else:
    print('\n⚠ No bedGraph files found!')
    print(f'  Checked: {BEDGRAPH_DIR}')
    print(f'  Looked for: *.bedGraph.gz, *.bedgraph.gz, *.bedGraph, *.txt')
    print(f'\n  Contents of directory:')
    if BEDGRAPH_DIR.exists():
        for f in sorted(BEDGRAPH_DIR.iterdir())[:10]:
            print(f'    {f.name}')
    else:
        print(f'    Directory does not exist!')

In [ ]:
# =============================================================================
# MATCH FILES TO METADATA
# =============================================================================
# Extract sample number from filename and join to metadata

def extract_sample_num(filepath):
    """Extract sample number from PN0634_XXX_... filename."""
    basename = os.path.basename(filepath)
    match = re.match(r'PN0634_(\d+)', basename)
    if match:
        return int(match.group(1))
    return None

file_lookup = {}
for fp in bedgraph_files:
    snum = extract_sample_num(fp)
    if snum is not None:
        file_lookup[snum] = fp

# Map files to metadata
metadata['bedgraph_path'] = metadata['sample_num'].map(file_lookup)

matched = metadata['bedgraph_path'].notna().sum()
missing = metadata[metadata['bedgraph_path'].isna()]

print(f'Matched: {matched}/{len(metadata)} samples')
if len(missing) > 0:
    print(f'\n⚠ Missing bedGraph files for {len(missing)} samples:')
    print(missing[['patient_id', 'sample_id', 'timepoint', 'response']].to_string(index=False))

## Section 4: Unzip, Clean Headers, Remove Non-Canonical Chromosomes

Per the TWIST Galaxy workflow:
- Extract `.gz` files
- Remove any header lines
- Remove non-canonical chromosomes (keep chr1-22, chrX, chrY only)
- Save as `.txt` text files for Galaxy upload

In [ ]:
# =============================================================================
# CLEAN BEDGRAPH FILES
# =============================================================================

os.makedirs(CLEANED_DIR, exist_ok=True)

def clean_bedgraph(input_path, output_path):
    """
    Unzip (if .gz), remove header lines and non-canonical chromosomes.
    Save as plain text file.
    
    Bismark bedGraph format: chr  start  end  methylation_percentage
    Header lines start with 'track' or don't begin with 'chr'.
    """
    lines_in = 0
    lines_out = 0
    lines_header = 0
    lines_noncanonical = 0
    
    # Open gzipped or plain
    if input_path.endswith('.gz'):
        infile = gzip.open(input_path, 'rt')
    else:
        infile = open(input_path, 'r')
    
    with infile, open(output_path, 'w') as out:
        for line in infile:
            lines_in += 1
            line = line.strip()
            if not line:
                continue
            
            # Skip header lines (track lines, browser lines, comments)
            if line.startswith('track') or line.startswith('browser') or line.startswith('#'):
                lines_header += 1
                continue
            
            # Extract chromosome (first field)
            chrom = line.split('\t')[0]
            
            # Skip non-canonical chromosomes
            if chrom not in CANONICAL_CHROMS:
                lines_noncanonical += 1
                continue
            
            out.write(line + '\n')
            lines_out += 1
    
    return lines_in, lines_out, lines_header, lines_noncanonical

print('Cleaning bedGraph files...\n')

clean_log = []
samples_to_process = metadata[metadata['bedgraph_path'].notna()].copy()

for i, (_, row) in enumerate(samples_to_process.iterrows()):
    input_path = row['bedgraph_path']
    # Output filename: PN0634_XXX_PatientID_timepoint.txt
    out_name = f"{row['sample_id']}_{row['patient_id']}_{row['timepoint']}.txt"
    output_path = str(CLEANED_DIR / out_name)
    
    lines_in, lines_out, lines_header, lines_noncanonical = clean_bedgraph(input_path, output_path)
    
    clean_log.append({
        'sample_id': row['sample_id'],
        'patient_id': row['patient_id'],
        'group': row['group'],
        'lines_in': lines_in,
        'lines_out': lines_out,
        'headers_removed': lines_header,
        'noncanonical_removed': lines_noncanonical,
        'cleaned_file': out_name
    })
    
    if (i + 1) % 10 == 0 or (i + 1) == len(samples_to_process):
        print(f'  Processed {i+1}/{len(samples_to_process)} files')

clean_df = pd.DataFrame(clean_log)
print(f'\n✓ Done. {len(clean_df)} files cleaned.')
print(f'  Headers removed (total): {clean_df["headers_removed"].sum()}')
print(f'  Non-canonical lines removed (total): {clean_df["noncanonical_removed"].sum()}')

In [ ]:
# Quick sanity check - peek at a cleaned file
example_file = sorted(CLEANED_DIR.glob('*.txt'))[0]
print(f'Example: {example_file.name}')
print('First 5 lines:')
with open(example_file) as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(f'  {line.strip()}')

# Check no headers remain
print(f'\nLast 3 lines:')
import subprocess
result = subprocess.run(['tail', '-3', str(example_file)], capture_output=True, text=True)
for line in result.stdout.strip().split('\n'):
    print(f'  {line}')

## Section 5: Organise into Group Folders for Galaxy

Create folders matching the 5 metilene runs.  
Each folder contains symlinks (not copies) to save disk space.  

```
galaxy_groups/
├── run1_resp_followup_vs_baseline/
│   ├── group1_followup/       ← test group
│   └── group2_baseline/       ← control group
├── run2_nonresp_followup_vs_baseline/
│   ├── group1_followup/
│   └── group2_baseline/
├── run3_resp_vs_nonresp_baseline/
│   ├── group1_resp_baseline/
│   └── group2_nonresp_baseline/
├── run4_resp_vs_nonresp_followup/
│   ├── group1_resp_followup/
│   └── group2_nonresp_followup/
└── run5_sensitivity_6wk_only/
    ├── resp_group1_followup/
    ├── resp_group2_baseline/
    ├── nonresp_group1_followup/
    └── nonresp_group2_baseline/
```

In [ ]:
# =============================================================================
# MAP CLEANED FILES TO METADATA
# =============================================================================

# Add cleaned file paths to metadata
metadata_matched = metadata[metadata['bedgraph_path'].notna()].copy()
metadata_matched['cleaned_file'] = metadata_matched.apply(
    lambda r: str(CLEANED_DIR / f"{r['sample_id']}_{r['patient_id']}_{r['timepoint']}.txt"),
    axis=1
)

# Verify all cleaned files exist
missing_cleaned = [r['cleaned_file'] for _, r in metadata_matched.iterrows() 
                   if not os.path.exists(r['cleaned_file'])]
if missing_cleaned:
    print(f'⚠ {len(missing_cleaned)} cleaned files missing!')
else:
    print(f'✓ All {len(metadata_matched)} cleaned files verified')

# Convenience subsets
resp_baseline = metadata_matched[
    (metadata_matched['response'] == 'responder') & (metadata_matched['timepoint'] == 'baseline')]
resp_followup = metadata_matched[
    (metadata_matched['response'] == 'responder') & (metadata_matched['timepoint'] == 'followup')]
nonresp_baseline = metadata_matched[
    (metadata_matched['response'] == 'nonresponder') & (metadata_matched['timepoint'] == 'baseline')]
nonresp_followup = metadata_matched[
    (metadata_matched['response'] == 'nonresponder') & (metadata_matched['timepoint'] == 'followup')]

print(f'\nGroup sizes:')
print(f'  Responder baseline:     {len(resp_baseline)}')
print(f'  Responder follow-up:    {len(resp_followup)}')
print(f'  Non-responder baseline: {len(nonresp_baseline)}')
print(f'  Non-responder follow-up:{len(nonresp_followup)}')

In [ ]:
# =============================================================================
# CREATE GROUP FOLDERS
# =============================================================================

def populate_folder(folder_path, file_df, use_copy=True):
    """
    Copy (or symlink) cleaned bedgraph files into a group folder.
    use_copy=True for portability (Galaxy upload), False for symlinks.
    """
    os.makedirs(folder_path, exist_ok=True)
    count = 0
    for _, row in file_df.iterrows():
        src = row['cleaned_file']
        dst = os.path.join(folder_path, os.path.basename(src))
        if use_copy:
            shutil.copy2(src, dst)
        else:
            if os.path.exists(dst):
                os.remove(dst)
            os.symlink(os.path.abspath(src), dst)
        count += 1
    return count

# --- Use copies (safer for Galaxy upload) ---
USE_COPY = False

print('Creating group folders...\n')

# RUN 1: Responders - follow-up vs baseline
run1 = GROUPS_DIR / 'run1_resp_followup_vs_baseline'
n1a = populate_folder(run1 / 'group1_followup', resp_followup, USE_COPY)
n1b = populate_folder(run1 / 'group2_baseline', resp_baseline, USE_COPY)
print(f'Run 1 (Resp FU vs BL):          G1={n1a}, G2={n1b}')

# RUN 2: Non-responders - follow-up vs baseline
run2 = GROUPS_DIR / 'run2_nonresp_followup_vs_baseline'
n2a = populate_folder(run2 / 'group1_followup', nonresp_followup, USE_COPY)
n2b = populate_folder(run2 / 'group2_baseline', nonresp_baseline, USE_COPY)
print(f'Run 2 (NonResp FU vs BL):       G1={n2a}, G2={n2b}')

# RUN 3: Responder baseline vs Non-responder baseline
run3 = GROUPS_DIR / 'run3_resp_vs_nonresp_baseline'
n3a = populate_folder(run3 / 'group1_resp_baseline', resp_baseline, USE_COPY)
n3b = populate_folder(run3 / 'group2_nonresp_baseline', nonresp_baseline, USE_COPY)
print(f'Run 3 (Resp vs NonResp BL):     G1={n3a}, G2={n3b}')

# RUN 4: Responder follow-up vs Non-responder follow-up
run4 = GROUPS_DIR / 'run4_resp_vs_nonresp_followup'
n4a = populate_folder(run4 / 'group1_resp_followup', resp_followup, USE_COPY)
n4b = populate_folder(run4 / 'group2_nonresp_followup', nonresp_followup, USE_COPY)
print(f'Run 4 (Resp vs NonResp FU):     G1={n4a}, G2={n4b}')

print(f'\n✓ 4 main runs ready in: {GROUPS_DIR}')

In [ ]:
# =============================================================================
# RUN 5: SENSITIVITY - 6-WEEK FOLLOW-UP ONLY
# =============================================================================
# Exclude patients whose on-treatment sample was taken at week 3
# (all other on-treatment samples were taken at week 6).

THREE_WEEK_PATIENTS = [
    "SJH30", "SJH39", "SJH53", "BH5", "BH6", "BH29", "BH42"
]

if THREE_WEEK_PATIENTS:
    # Filter to 6-week only
    six_wk = metadata_matched[~metadata_matched['patient_id'].isin(THREE_WEEK_PATIENTS)]
    
    resp_bl_6wk = six_wk[(six_wk['response'] == 'responder') & (six_wk['timepoint'] == 'baseline')]
    resp_fu_6wk = six_wk[(six_wk['response'] == 'responder') & (six_wk['timepoint'] == 'followup')]
    nonresp_bl_6wk = six_wk[(six_wk['response'] == 'nonresponder') & (six_wk['timepoint'] == 'baseline')]
    nonresp_fu_6wk = six_wk[(six_wk['response'] == 'nonresponder') & (six_wk['timepoint'] == 'followup')]
    
    run5 = GROUPS_DIR / 'run5_sensitivity_6wk_only'
    n5a = populate_folder(run5 / 'resp_group1_followup', resp_fu_6wk, USE_COPY)
    n5b = populate_folder(run5 / 'resp_group2_baseline', resp_bl_6wk, USE_COPY)
    n5c = populate_folder(run5 / 'nonresp_group1_followup', nonresp_fu_6wk, USE_COPY)
    n5d = populate_folder(run5 / 'nonresp_group2_baseline', nonresp_bl_6wk, USE_COPY)
    
    print(f'Run 5 Sensitivity (6-week only):')
    print(f'  Excluded {len(THREE_WEEK_PATIENTS)} patients with 3-week follow-up')
    print(f'  Resp: G1(FU)={n5a}, G2(BL)={n5b}')
    print(f'  NonResp: G1(FU)={n5c}, G2(BL)={n5d}')
else:
    print('⚠ THREE_WEEK_PATIENTS list is empty.')
    print('  Fill in the patient IDs above, then re-run this cell.')
    print('  Skipping Run 5 for now - runs 1-4 are ready.')

## Section 6: Summary and Export Log

In [ ]:
# =============================================================================
# SAVE METADATA AND CLEANING LOG
# =============================================================================

# Save full metadata with group assignments
metadata_matched.to_csv(OUTPUT_DIR / 'metilene_metadata.csv', index=False)
print(f'✓ Saved: metilene_metadata.csv')

# Save cleaning log
clean_df.to_csv(OUTPUT_DIR / 'cleaning_log.csv', index=False)
print(f'✓ Saved: cleaning_log.csv')

# Print final directory structure
print(f'\n{"="*60}')
print('FINAL OUTPUT STRUCTURE')
print(f'{"="*60}\n')

for root, dirs, files in os.walk(GROUPS_DIR):
    level = root.replace(str(GROUPS_DIR), '').count(os.sep)
    indent = '  ' * level
    dirname = os.path.basename(root)
    n_files = len([f for f in files if f.endswith('.txt')])
    if n_files > 0:
        print(f'{indent}{dirname}/ ({n_files} files)')
    else:
        print(f'{indent}{dirname}/')

---

## 🛑 STOP - Upload to Galaxy and Run Metilene

**For each run:**
1. Upload the `group1_*` files as **Input group 1** (test/disease group)
2. Upload the `group2_*` files as **Input group 2** (control group)
3. Set file type to **bedgraph**, genome to **Hg38 (2013)**
4. Run metilene with default parameters
5. **Do NOT** upload a CpG island BED file to regions of interest
6. Download the `metilene qval <0.05` output

**Positive mean methylation difference = upregulated in Group 1 (test)**

When all 5 runs are complete, continue below.

---

## Section 7: Post-Galaxy - Load and Organise Metilene Results

After running metilene on Galaxy, download the results and place them in:
```
results/metilene/galaxy_results/
├── run1_resp_dmrs.txt
├── run2_nonresp_dmrs.txt
├── run3_baseline_dmrs.txt
├── run4_followup_dmrs.txt
└── run5_sensitivity_dmrs.txt
```

In [ ]:
# =============================================================================
# LOAD METILENE RESULTS (run after Galaxy)
# =============================================================================

GALAXY_RESULTS_DIR = OUTPUT_DIR / 'galaxy_results'

# Metilene output columns (from TWIST doc):
# chr  start  end  q-value  mean_meth_diff  #CpGs  mean_g1  mean_g2  ...
METILENE_COLS = ['chr', 'start', 'end', 'qval', 'mean_meth_diff', 
                 'num_cpgs', 'mean_g1', 'mean_g2']

def load_metilene_result(filepath, label):
    """Load a metilene qval<0.05 output file."""
    if not os.path.exists(filepath):
        print(f'  ⚠ {label}: file not found ({filepath})')
        return None
    df = pd.read_csv(filepath, sep='\t', header=None)
    # Assign column names (metilene output may have more columns)
    df.columns = METILENE_COLS[:len(df.columns)] + \
                 [f'col_{i}' for i in range(len(df.columns) - len(METILENE_COLS))] \
                 if len(df.columns) > len(METILENE_COLS) else METILENE_COLS[:len(df.columns)]
    print(f'  {label}: {len(df)} DMRs (q<0.05)')
    return df

results = {}
print('Loading metilene results...\n')

result_files = {
    'run1_resp': 'run1_resp_dmrs.txt',
    'run2_nonresp': 'run2_nonresp_dmrs.txt',
    'run3_baseline': 'run3_baseline_dmrs.txt',
    'run4_followup': 'run4_followup_dmrs.txt',
    'run5a_resp_6wk': 'run5a_resp_6wk_dmrs.txt',
    'run5b_nonresp_6wk': 'run5b_nonresp_6wk_dmrs.txt',
}

for key, fname in result_files.items():
    df = load_metilene_result(GALAXY_RESULTS_DIR / fname, key)
    if df is not None:
        results[key] = df

if not results:
    print('\n⚠ No results loaded yet. Run Galaxy first, then place files in:')
    print(f'   {GALAXY_RESULTS_DIR}/')

In [ ]:
# =============================================================================
# ANNOTATE DMRs WITH ENSEMBL GENE IDs (pure pandas, no bedtools needed)
# =============================================================================

ENSEMBL_BED = PROJECT_DIR / 'database_files' / 'metadata' / 'Ensembl_Gene_ID_For_Annotation.txt'

genes = pd.read_csv(ENSEMBL_BED, sep='\t', header=None, 
                     names=['gene_chr', 'gene_start', 'gene_end', 'gene_name'])

print(f'Loaded {len(genes)} gene annotations')

for key, df in results.items():
    if df is None or len(df) == 0:
        continue
    
    gene_names = []
    for _, dmr in df.iterrows():
        # Find overlapping genes: same chr, and ranges overlap
        overlap = genes[
            (genes['gene_chr'] == dmr['chr']) &
            (genes['gene_start'] < dmr['end']) &
            (genes['gene_end'] > dmr['start'])
        ]
        if len(overlap) > 0:
            gene_names.append('; '.join(sorted(overlap['gene_name'].dropna().unique())))
        else:
            gene_names.append('')
    
    df['gene_name'] = gene_names
    n_annotated = (df['gene_name'] != '').sum()
    print(f'{key}: {n_annotated}/{len(df)} DMRs annotated with gene names')

In [ ]:
# =============================================================================
# SORT AND CREATE SPREADSHEET (per TWIST doc)
# =============================================================================
# For each run, create the 4 sheets:
#   1. Mean Meth Diff Sort
#   2. qval Sort (positive diff only - upregulated in test group)
#   3. qval and FC Sort (add fold change: mean_g1 / mean_g2)
#   4. qval and FC Sort Ensembl ID (add after bedtools intersect on Galaxy)

SPREADSHEET_DIR = OUTPUT_DIR / 'spreadsheets'
os.makedirs(SPREADSHEET_DIR, exist_ok=True)

for key, df in results.items():
    if df is None or len(df) == 0:
        continue
    
    outpath = SPREADSHEET_DIR / f'{key}_metilene_results.xlsx'
    
    with pd.ExcelWriter(outpath, engine='openpyxl') as writer:
        # Sheet 1: Sort by mean methylation difference (largest to smallest)
        sheet1 = df.sort_values('mean_meth_diff', ascending=False)
        sheet1.to_excel(writer, sheet_name='Mean Meth Diff Sort', index=False)
        
        # Sheet 2: Positive diff only (up in test group), sorted by qval
        sheet2 = df[df['mean_meth_diff'] > 0].sort_values('qval', ascending=True)
        sheet2.to_excel(writer, sheet_name='qval Sort', index=False)
        
        # Sheet 3: Add fold change
        sheet3 = sheet2.copy()
        sheet3['fold_change'] = sheet3['mean_g1'] / sheet3['mean_g2']
        sheet3 = sheet3.sort_values('qval', ascending=True)
        sheet3 = sheet3[sheet3['fold_change'] >= 2]
        sheet3.to_excel(writer, sheet_name='qval and FC Sort', index=False)
        
        # Sheet 4: Placeholder for Ensembl annotation (after bedtools intersect)
        sheet4 = sheet3.copy()
        if 'gene_name' in df.columns:
            sheet4 = sheet4.merge(df[['chr', 'start', 'end', 'gene_name']].drop_duplicates(),
                                  on=['chr', 'start', 'end'], how='left')
        sheet4.to_excel(writer, sheet_name='qval FC Ensembl ID', index=False)
    
    print(f'✓ {key}: saved {outpath.name}')

print(f'\nSpreadsheets saved to: {SPREADSHEET_DIR}')

---

## Done

**Output structure:**
```
results/metilene/
├── cleaned_bedgraphs/           ← cleaned .txt files for Galaxy
├── galaxy_groups/               ← organised by run/group
│   ├── run1_resp_.../
│   ├── run2_nonresp_.../
│   ├── run3_..._baseline/
│   ├── run4_..._followup/
│   └── run5_sensitivity_.../
├── galaxy_results/              ← download metilene output here
├── spreadsheets/                ← Excel files per TWIST doc format
├── bed_for_annotation/          ← BED files for bedtools intersect
├── metilene_metadata.csv
└── cleaning_log.csv
```